# Cellpose-SAM

Fine-tunes the ImageGrains 2.0 model on the tile archive, once per fold, and
pools the four runs at the end.

Cellpose-SAM is locked to 256 px crops, so the crops come out of the archive at
384 and cellpose takes its 256 from inside them. The margin means a rotation
never pulls in empty corners.

Crops only come from positions where the whole 384 window is inspected ground,
so the label raster is the full truth for every pixel the net sees. Hillside and
anything past the image edge is never used.

Every fold writes `results/cellpose_sam_<fold>.json` as soon as it is done, and
a fold whose file is already there is skipped.

In [ ]:
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from cellpose import models, train

warnings.filterwarnings("ignore")
root = Path.cwd()
while not (root / "config.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import config
from common import tiles, crops, evaluate, folds

FOLDS = ["fold1", "fold2", "fold3", "fold4"]
EPOCHS = 150
PER_EPOCH = 250              # crops drawn per epoch, this drives the runtime
TEST_PER_EPOCH = 100         # validation loss, cellpose would use all of them
BATCH = 1
LR = 1e-5
WEIGHT_DECAY = 0.1
SCALE_RANGE = 0.5            # 0.75 to 1.25
SAVE_EVERY = 20
SAMPLE = 50                  # tiles used for the checkpoint curve
BASE = config.MODELS / "IG2_full_set_cp_SAM"

print("gpu:", torch.cuda.get_device_name(0))
print("base model:", BASE.name, "-", BASE.exists())
print(f"{len(FOLDS)} folds x {EPOCHS * PER_EPOCH} crops, "
      f"about {len(FOLDS) * EPOCHS * PER_EPOCH * 0.50 / 3600:.0f} h plus scoring")

## Tiles per fold

Validation is one fold, training is everything else except the test block. The
test block stays untouched until the very end.

In [ ]:
idx = pd.DataFrame(tiles.read_index())
print(idx.groupby("fold").agg(tiles=("name", "size"), boulders=("n_inner", "sum")))
print(f"\ntest block held back: {int((idx.fold == 'test').sum())} tiles")

## Crops

One crop centred on each boulder plus 30 % random corridor positions. Tiles
without labels are left out, their background is already in every other crop.

In [ ]:
def crop_sets(fold):
    train_names = idx.loc[idx.fold.isin(folds.training_folds(fold))
                          & (idx.n_inner > 0), "name"].tolist()
    val_names = idx.loc[(idx.fold == fold) & (idx.n_inner > 0), "name"].tolist()
    xtr, ytr = crops.build(train_names, seed=0, verbose=False)
    xva, yva = crops.build(val_names, seed=1, verbose=False)
    return xtr, ytr, xva, yva, val_names

## What the crops look like

In [ ]:
xtr, ytr, xva, yva, val_names = crop_sets(FOLDS[0])
print(f"{FOLDS[0]}: {len(xtr)} training crops, {len(xva)} validation crops, "
      f"{len(val_names)} validation tiles")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (a, b) in zip(axes.ravel(), zip(xtr[:8], ytr[:8])):
    lo, hi = np.percentile(a[a > 0], config.PERCENTILE)
    ax.imshow(np.clip((a - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.contour(b > 0, levels=[0.5], colors="red", linewidths=0.8)
    ax.set_title(f"{int(b.max())} boulders", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Train and score

Checkpoints every 20 epochs. Cellpose has no early stopping, so all of them get
scored on a sample of the validation tiles and the best one is then scored on
all of them.

Scoring runs on whole tiles, not crops, because that is how it gets applied.
Only objects whose centre falls inside the corridor and inside the tile's own
inner region count, so nothing is scored twice and nothing on ground that was
never inspected.

In [ ]:
def train_fold(fold, xtr, ytr, xva, yva):
    out = config.MODELS / f"cellpose_sam_{fold}"
    out.mkdir(parents=True, exist_ok=True)
    net = models.CellposeModel(gpu=True, pretrained_model=str(BASE)).net
    _, train_loss, test_loss = train.train_seg(
        net,
        train_data=xtr, train_labels=ytr,
        test_data=xva, test_labels=yva,
        batch_size=BATCH, n_epochs=EPOCHS, nimg_per_epoch=PER_EPOCH,
        nimg_test_per_epoch=TEST_PER_EPOCH,
        learning_rate=LR, weight_decay=WEIGHT_DECAY,
        scale_range=SCALE_RANGE, bsize=256,
        normalize={"percentile": list(config.PERCENTILE)},
        min_train_masks=1,
        save_path=str(out), save_every=SAVE_EVERY, save_each=True,
        model_name=fold)
    return out, list(train_loss), list(test_loss)


def predict_tile(model, name):
    img, true, valid = crops.read_tile(name)
    pred, _, _ = model.eval(img.astype(np.float32), batch_size=8,
                            normalize={"percentile": list(config.PERCENTILE)},
                            flow_threshold=0.4, cellprob_threshold=0.0,
                            min_size=15)
    keep = valid.astype(bool) & evaluate.inner_mask(img.shape)
    return evaluate.restrict(pred.astype(np.uint16), keep), evaluate.restrict(true, keep)


def score_checkpoint(ckpt, names):
    model = models.CellposeModel(gpu=True, pretrained_model=str(ckpt))
    rows = [evaluate.score(*predict_tile(model, n)) for n in names]
    del model
    torch.cuda.empty_cache()
    return evaluate.summarise(rows)

In [ ]:
config.RESULTS.mkdir(parents=True, exist_ok=True)
started = time.time()

for fold in FOLDS:
    print(f"\n=== {fold} ===", flush=True)
    done = config.RESULTS / f"cellpose_sam_{fold}.json"
    if done.exists():
        print("already scored, skipped", flush=True)
        continue

    t0 = time.time()
    if fold == FOLDS[0]:
        a, b, c, d, names = xtr, ytr, xva, yva, val_names
    else:
        a, b, c, d, names = crop_sets(fold)
    print(f"{len(a)} training crops, {len(c)} validation crops, "
          f"{len(names)} validation tiles", flush=True)

    out, train_loss, test_loss = train_fold(fold, a, b, c, d)
    print(f"trained in {(time.time() - t0) / 3600:.2f} h", flush=True)

    curve = []
    for ck in sorted((out / "models").glob(f"{fold}*")):
        s = score_checkpoint(ck, names[:SAMPLE])
        s["checkpoint"] = ck.name
        curve.append(s)
        print(f"  {ck.name:28} AP {s['ap']:.3f}  P {s['precision']:.3f}  "
              f"R {s['recall']:.3f}", flush=True)

    best = max(curve, key=lambda s: s["ap"])["checkpoint"]
    final = score_checkpoint(out / "models" / best, names)
    print(f"  best {best}: AP {final['ap']:.3f} on all {len(names)} tiles", flush=True)

    json.dump(dict(model="cellpose_sam", fold=fold, base=BASE.name,
                   epochs=EPOCHS, per_epoch=PER_EPOCH, batch=BATCH, lr=LR,
                   scale_range=SCALE_RANGE, res=config.RES, crop=crops.SIZE,
                   bsize=256, n_train_crops=len(a), n_val_crops=len(c),
                   n_val_tiles=len(names), best_checkpoint=best,
                   hours=round((time.time() - t0) / 3600, 2),
                   train_loss=train_loss, test_loss=test_loss,
                   curve=curve, **final),
              open(done, "w"), indent=1)

print(f"\nall folds done in {(time.time() - started) / 3600:.1f} h")

## Results

Every boulder in the four folds was predicted exactly once, by a model that
never saw it. The pooled numbers add up TP, FP and FN across the folds rather
than averaging the APs, which would give small folds too much weight.

In [ ]:
runs = [json.loads(f.read_text()) for f in
        sorted(config.RESULTS.glob("cellpose_sam_fold*.json"))]

keep = ["fold", "tp", "fp", "fn", "ap", "precision", "recall", "mean_iou",
        "d50_pred", "d50_true", "d84_pred", "d84_true", "size_bias", "hours"]
per_fold = pd.DataFrame([{k: r[k] for k in keep} for r in runs]).set_index("fold")

tp, fp, fn = (int(per_fold[c].sum()) for c in ("tp", "fp", "fn"))
dp = np.concatenate([r["d_pred"] for r in runs])
dt = np.concatenate([r["d_true"] for r in runs])

per_fold.loc["pooled"] = {
    "tp": tp, "fp": fp, "fn": fn,
    "ap": tp / (tp + fp + fn),
    "precision": tp / (tp + fp),
    "recall": tp / (tp + fn),
    "mean_iou": np.average(per_fold.mean_iou, weights=per_fold.tp),
    "d50_pred": np.median(dp), "d84_pred": np.percentile(dp, 84),
    "d50_true": np.median(dt), "d84_true": np.percentile(dt, 84),
    "size_bias": np.average(per_fold.size_bias, weights=per_fold.tp),
    "hours": per_fold["hours"].sum()}
per_fold.round(3)

In [ ]:
ap = per_fold.ap.drop("pooled")
print(f"AP per fold {ap.min():.3f} to {ap.max():.3f}, pooled {per_fold.ap['pooled']:.3f}")
print(f"precision {per_fold.precision['pooled']:.3f}, "
      f"recall {per_fold.recall['pooled']:.3f}, "
      f"mean IoU {per_fold.mean_iou['pooled']:.3f}")
print(f"D50 {per_fold.d50_pred['pooled']:.2f} m predicted against "
      f"{per_fold.d50_true['pooled']:.2f} m mapped, "
      f"bias {per_fold.size_bias['pooled']:+.2f} m")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

for r in runs:
    axes[0].plot(r["train_loss"], lw=1, label=r["fold"])
    t = np.array(r["test_loss"], float)
    axes[1].plot(np.where(t > 0, t, np.nan), lw=1.4, label=r["fold"])
    axes[2].plot([c["ap"] for c in r["curve"]], "o-", lw=1, label=r["fold"])

axes[0].set_title("training loss")
axes[1].set_title("validation loss")
axes[2].set_title("AP per checkpoint")
axes[2].axhline(0.16, color="k", ls="--", lw=1)
for ax, x in zip(axes, ("epoch", "epoch", "checkpoint")):
    ax.set_xlabel(x)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(per_fold.d50_true.drop("pooled"), per_fold.d50_pred.drop("pooled"),
                label="D50")
axes[0].scatter(per_fold.d84_true.drop("pooled"), per_fold.d84_pred.drop("pooled"),
                label="D84")
lim = [3, 9]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set_xlim(lim)
axes[0].set_ylim(lim)
axes[0].set_xlabel("mapped (m)")
axes[0].set_ylabel("predicted (m)")
axes[0].legend()

axes[1].bar(ap.index, ap.values)
axes[1].axhline(per_fold.ap["pooled"], color="k", ls="--", lw=1)
axes[1].set_ylabel("AP")
plt.tight_layout()
plt.show()

## Predictions against the hand drawn outlines

In [ ]:
model = models.CellposeModel(
    gpu=True,
    pretrained_model=str(config.MODELS / f"cellpose_sam_{FOLDS[0]}" / "models"
                         / runs[0]["best_checkpoint"]))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, name in zip(axes.ravel(), val_names[:6]):
    img, _, _ = crops.read_tile(name)
    p, t = predict_tile(model, name)
    lo, hi = np.percentile(img[img > 0], config.PERCENTILE)
    ax.imshow(np.clip((img - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.contour(t > 0, levels=[0.5], colors="red", linewidths=0.8)
    ax.contour(p > 0, levels=[0.5], colors="lime", linewidths=0.8)
    ax.set_title(f"{name}  mapped {int(t.max())}  found {int(p.max())}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()